<a href="https://colab.research.google.com/github/srinivasulu-2026/my-first-repo/blob/main/Swiggy_workshop_17th_May_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛵 How Swiggy Tracks 10M Orders Daily
### A hands-on workshop with NumPy, Pandas & Google Colab

In [ ]:
# Run this once — it builds swiggy_orders.csv in the Colab session
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

rng = np.random.default_rng(42)
N = 3000

cities = ["Bengaluru", "Hyderabad", "Mumbai", "Delhi", "Chennai", "Pune", "Kolkata", "Ahmedabad"]
city_weights = [0.22, 0.16, 0.18, 0.15, 0.10, 0.08, 0.06, 0.05]
categories = ["South Indian", "North Indian", "Chinese", "Pizza", "Burgers", "Desserts", "Beverages", "Healthy Bowls"]
cat_weights = [0.18, 0.20, 0.14, 0.12, 0.10, 0.10, 0.08, 0.08]

restaurants = {
    "South Indian": ["Sri Udupi Palace", "Adyar Ananda Bhavan", "MTR", "Brahmin's Coffee Bar", "Dosa Plaza"],
    "North Indian": ["Sagar Ratna", "Haldiram's", "Rajdhani Thali", "Veg Treat", "Punjabi Tadka"],
    "Chinese": ["Wok in the Clouds", "Mainland Veg China", "Bowl Co.", "Chung Wah Veg"],
    "Pizza": ["Pizza Hut Veg", "La Pino'z", "Oven Story Veg", "Pizza Bakery"],
    "Burgers": ["Burger Singh Veg", "Veg Stop Burgers", "Plantman Burgers"],
    "Desserts": ["Theobroma", "Bakingo", "Ovenfresh", "Sweet Truth"],
    "Beverages": ["Chai Point", "Boba Bhai", "Blue Tokai", "Third Wave Coffee"],
    "Healthy Bowls": ["Salad Days", "The Bowl Company", "Greenr Cafe", "Eat.Fit"],
}
items_by_cat = {
    "South Indian": ["Masala Dosa", "Idli Sambar", "Vada", "Uttapam", "Filter Coffee", "Pongal", "Rava Dosa"],
    "North Indian": ["Paneer Butter Masala", "Dal Makhani", "Veg Biryani", "Chole Bhature", "Aloo Paratha", "Palak Paneer", "Mushroom Masala"],
    "Chinese": ["Veg Hakka Noodles", "Veg Manchurian", "Veg Fried Rice", "Schezwan Noodles", "Spring Rolls", "Chilli Paneer"],
    "Pizza": ["Margherita", "Farmhouse", "Paneer Tikka Pizza", "Veggie Supreme", "Cheese Burst Veg", "Mexican Green Wave"],
    "Burgers": ["Aloo Tikki Burger", "Paneer Burger", "Veg Whopper", "Crispy Veg Burger"],
    "Desserts": ["Choco Lava Cake", "Gulab Jamun", "Brownie", "Rasmalai", "Tiramisu (eggless)", "Cheesecake (eggless)"],
    "Beverages": ["Masala Chai", "Cold Coffee", "Mango Smoothie", "Lemonade", "Hot Chocolate"],
    "Healthy Bowls": ["Quinoa Bowl", "Buddha Bowl", "Falafel Bowl", "Khichdi Bowl", "Veg Poke Bowl"],
}
price_range = {"South Indian":(80,250),"North Indian":(150,450),"Chinese":(140,380),"Pizza":(200,600),
               "Burgers":(120,320),"Desserts":(100,350),"Beverages":(60,220),"Healthy Bowls":(180,420)}
payment_modes = ["UPI","Card","Cash","Wallet"]; pay_weights=[0.55,0.22,0.13,0.10]
statuses = ["Delivered","Cancelled"]; status_w=[0.93,0.07]

rows = []
base_time = datetime(2024, 11, 1)
for i in range(N):
    city = rng.choice(cities, p=city_weights)
    category = rng.choice(categories, p=cat_weights)
    restaurant = rng.choice(restaurants[category])
    item = rng.choice(items_by_cat[category])
    lo, hi = price_range[category]
    item_price = float(rng.integers(lo, hi+1))
    quantity = int(rng.choice([1,1,1,2,2,3], p=[0.45,0.10,0.05,0.25,0.10,0.05]))
    delivery_fee = float(rng.integers(20,60)) + (10 if city in ("Mumbai","Bengaluru") else 0)
    discount = float(rng.choice([0,0,0,20,30,50,75,100], p=[0.45,0.10,0.05,0.10,0.10,0.10,0.06,0.04]))
    base_d = {"Bengaluru":38,"Hyderabad":32,"Mumbai":42,"Delhi":36,"Chennai":30,"Pune":33,"Kolkata":34,"Ahmedabad":29}[city]
    delivery_time = int(np.clip(rng.normal(base_d, 8), 12, 95))
    rating = float(np.round(np.clip(rng.normal(4.2, 0.6), 1.0, 5.0), 1))
    status = rng.choice(statuses, p=status_w)
    payment_mode = rng.choice(payment_modes, p=pay_weights)
    order_time = base_time + timedelta(days=int(rng.integers(0,30)), hours=int(rng.integers(8,24)), minutes=int(rng.integers(0,60)))
    rows.append({"order_id":100000+i,"order_time":order_time,"city":city,"restaurant":restaurant,
                 "category":category,"item":item,"item_price":item_price,"quantity":quantity,
                 "delivery_fee":delivery_fee,"discount":discount,"payment_mode":payment_mode,
                 "delivery_time_min":delivery_time,"rating":rating,"status":status})

df_raw = pd.DataFrame(rows)

# Inject realistic mess
null_idx = rng.choice(df_raw.index, size=int(0.15*len(df_raw)), replace=False)
df_raw.loc[null_idx, "rating"] = np.nan
null_idx2 = rng.choice(df_raw.index, size=int(0.03*len(df_raw)), replace=False)
df_raw.loc[null_idx2, "delivery_time_min"] = np.nan
null_idx3 = rng.choice(df_raw.index, size=int(0.02*len(df_raw)), replace=False)
df_raw.loc[null_idx3, "payment_mode"] = np.nan
cancelled = df_raw["status"] == "Cancelled"
df_raw.loc[cancelled & (rng.random(len(df_raw)) < 0.7), "delivery_time_min"] = np.nan
df_raw.loc[cancelled & (rng.random(len(df_raw)) < 0.8), "rating"] = np.nan

# Duplicates + casing typos
dup_sample = df_raw.sample(n=25, random_state=7)
df_raw = pd.concat([df_raw, dup_sample], ignore_index=True)
typo_idx = rng.choice(df_raw.index, size=15, replace=False)
df_raw.loc[typo_idx, "city"] = df_raw.loc[typo_idx, "city"].str.lower()
df_raw = df_raw.sample(frac=1.0, random_state=11).reset_index(drop=True)

df_raw.to_csv("swiggy_orders.csv", index=False)
print(f"✅ Dataset created: swiggy_orders.csv ({len(df_raw)} rows)")

✅ Dataset created: swiggy_orders.csv (3025 rows)


In [ ]:
import pandas as pd
df = pd.read_csv("swiggy_orders.csv",parse_dates=["order_time"])
df.head()

,order_id,order_time,city,restaurant,category,item,item_price,quantity,delivery_fee,discount,payment_mode,delivery_time_min,rating,status
0,101756,2024-11-27 22:23:00,Chennai,Ovenfresh,Desserts,Brownie,190.0,2,52.0,0.0,UPI,21.0,4.7,Delivered
1,101396,2024-11-30 09:00:00,Pune,Haldiram's,North Indian,Dal Makhani,243.0,1,57.0,0.0,UPI,27.0,3.3,Delivered
2,100270,2024-11-08 22:32:00,Hyderabad,Veg Treat,North Indian,Chole Bhature,168.0,1,27.0,0.0,UPI,NaN,NaN,Cancelled
3,102032,2024-11-02 19:57:00,Bengaluru,Haldiram's,North Indian,Palak Paneer,202.0,2,33.0,100.0,UPI,40.0,4.7,Delivered
4,101867,2024-11-08 10:05:00,Pune,Oven Story Veg,Pizza,Margherita,360.0,2,47.0,30.0,Card,35.0,4.8,Delivered


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3025 entries, 0 to 3024
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   order_id           3025 non-null   int64         
 1   order_time         3025 non-null   datetime64[ns]
 2   city               3025 non-null   object        
 3   restaurant         3025 non-null   object        
 4   category           3025 non-null   object        
 5   item               3025 non-null   object        
 6   item_price         3025 non-null   float64       
 7   quantity           3025 non-null   int64         
 8   delivery_fee       3025 non-null   float64       
 9   discount           3025 non-null   float64       
 10  payment_mode       2962 non-null   object        
 11  delivery_time_min  2799 non-null   float64       
 12  rating             2427 non-null   float64       
 13  status             3025 non-null   object        
dtypes: datet

In [ ]:
df.describe()

,order_id,order_time,item_price,quantity,delivery_fee,discount,delivery_time_min,rating
count,3025.000000,3025,3025.000000,3025.000000,3025.000000,3025.000000,2799.000000,2427.000000
mean,101500.658182,2024-11-15 23:58:10.552065792,252.960661,1.445620,43.657851,18.094215,34.933190,4.184508
min,100000.000000,2024-11-01 08:03:00,60.000000,1.000000,20.000000,0.000000,12.000000,2.100000
25%,100753.000000,2024-11-08 08:56:00,175.000000,1.000000,34.000000,0.000000,29.000000,3.800000
50%,101498.000000,2024-11-15 23:00:00,240.000000,1.000000,44.000000,0.000000,35.000000,4.200000
75%,102248.000000,2024-11-23 13:52:00,320.000000,2.000000,54.000000,30.000000,41.000000,4.600000
max,102999.000000,2024-11-30 23:50:00,595.000000,3.000000,69.000000,100.000000,66.000000,5.000000
std,864.842803,NaN,103.485114,0.586247,12.462309,27.527272,8.952669,0.571015


In [ ]:
df.shape

(3025, 14)

In [ ]:
df["city"].head()

,city
0,Chennai
1,Pune
2,Hyderabad
3,Bengaluru
4,Pune


In [ ]:
df[["order_id","city","quantity"]].head()

,order_id,city,quantity
0,101756,Chennai,2
1,101396,Pune,1
2,100270,Hyderabad,1
3,102032,Bengaluru,2
4,101867,Pune,2


Show the first 3 rows of just - restaurant, category and item_price?\
.describe() for delivery_time_min?

In [ ]:
print(df[["restaurant","category","item_price"]].head(3))
print(df["delivery_time_min"].describe())

   restaurant      category  item_price
0   Ovenfresh      Desserts       190.0
1  Haldiram's  North Indian       243.0
2   Veg Treat  North Indian       168.0
count    2799.000000
mean       34.933190
std         8.952669
min        12.000000
25%        29.000000
50%        35.000000
75%        41.000000
max        66.000000
Name: delivery_time_min, dtype: float64


How many orders happened in Bengaluru?

In [ ]:
bengaluru_mask = df["city"] == "Bengaluru"
df[bengaluru_mask].shape[0]

632

Find the pizza orders in Mumbai over 400 rupees?

In [ ]:
mask = (df["city"] == "Mumbai") & (df["category"] == "Pizza") & (df["item_price"] > 400)
df[mask]["order_id"].count()

np.int64(24)

Cancelled orders OR orders with rating below 3?

In [ ]:
bad_experience_mask = df[(df["status"] == "Cancelled") | (df["rating"] < 3)]
print(f"Bad_experience:{len(bad_experience_mask)}")
bad_experience_mask[["order_id", "city", "status","rating"]].head()

Bad_experience:260


,order_id,city,status,rating
2,100270,Hyderabad,Cancelled,NaN
22,100372,Mumbai,Cancelled,NaN
41,100726,Delhi,Cancelled,NaN
50,101100,Mumbai,Delivered,2.9
69,100592,Chennai,Cancelled,NaN


orders from top 3 (Bengaluru, Mumbai, Delhi)?\
Mid-priced items (200 to 400 Rs)?

In [ ]:
metros=df[df["city"].isin(["Bengaluru","Mumbai","Delhi"])]
print(metros["order_id"].count())

1598


In [ ]:
df[df["item_price"].between(200,400)].head()

,order_id,order_time,city,restaurant,category,item,item_price,quantity,delivery_fee,discount,payment_mode,delivery_time_min,rating,status
1,101396,2024-11-30 09:00:00,Pune,Haldiram's,North Indian,Dal Makhani,243.0,1,57.0,0.0,UPI,27.0,3.3,Delivered
3,102032,2024-11-02 19:57:00,Bengaluru,Haldiram's,North Indian,Palak Paneer,202.0,2,33.0,100.0,UPI,40.0,4.7,Delivered
4,101867,2024-11-08 10:05:00,Pune,Oven Story Veg,Pizza,Margherita,360.0,2,47.0,30.0,Card,35.0,4.8,Delivered
5,101688,2024-11-11 21:47:00,Ahmedabad,Salad Days,Healthy Bowls,Falafel Bowl,380.0,2,47.0,0.0,UPI,31.0,NaN,Delivered
6,102211,2024-11-06 12:36:00,Pune,Rajdhani Thali,North Indian,Aloo Paratha,388.0,1,49.0,0.0,UPI,29.0,4.0,Delivered


Top 5 expensive single items?\
Sort by multiple columns - city ascending, then price descending within each city?

In [ ]:
df.sort_values("item_price",ascending=False).head(5)[["order_id","city","item","item_price"]]

,order_id,city,item,item_price
1016,102409,Hyderabad,Mexican Green Wave,595.0
2150,101178,Mumbai,Cheese Burst Veg,593.0
2773,101761,Kolkata,Margherita,593.0
156,100844,Bengaluru,Paneer Tikka Pizza,592.0
1804,100905,Mumbai,Mexican Green Wave,590.0


In [ ]:
df.sort_values(["city","item_price"],ascending=[True,False]).head(10)

,order_id,order_time,city,restaurant,category,item,item_price,quantity,delivery_fee,discount,payment_mode,delivery_time_min,rating,status
2742,102370,2024-11-02 19:36:00,Ahmedabad,La Pino'z,Pizza,Margherita,550.0,2,42.0,100.0,Cash,21.0,4.7,Delivered
2892,102667,2024-11-24 22:49:00,Ahmedabad,Pizza Bakery,Pizza,Mexican Green Wave,543.0,2,57.0,0.0,Card,40.0,3.4,Delivered
259,102587,2024-11-08 22:03:00,Ahmedabad,Pizza Bakery,Pizza,Margherita,508.0,1,56.0,75.0,UPI,20.0,NaN,Delivered
1416,100020,2024-11-14 11:20:00,Ahmedabad,Pizza Hut Veg,Pizza,Mexican Green Wave,496.0,1,52.0,30.0,Cash,31.0,4.9,Delivered
1069,100645,2024-11-04 17:57:00,Ahmedabad,Oven Story Veg,Pizza,Veggie Supreme,493.0,2,54.0,20.0,Wallet,23.0,4.9,Delivered
3011,102594,2024-11-10 22:29:00,Ahmedabad,La Pino'z,Pizza,Margherita,471.0,2,21.0,75.0,UPI,27.0,5.0,Delivered
2184,100742,2024-11-16 21:35:00,Ahmedabad,Oven Story Veg,Pizza,Margherita,454.0,2,30.0,0.0,UPI,36.0,3.1,Delivered
1276,102050,2024-11-18 17:21:00,Ahmedabad,Rajdhani Thali,North Indian,Mushroom Masala,441.0,1,43.0,30.0,Card,19.0,NaN,Delivered
975,100641,2024-11-23 15:12:00,Ahmedabad,Pizza Hut Veg,Pizza,Mexican Green Wave,436.0,1,52.0,30.0,Cash,20.0,3.8,Delivered
897,100903,2024-11-13 19:02:00,Ahmedabad,Haldiram's,North Indian,Dal Makhani,424.0,2,57.0,0.0,UPI,28.0,3.7,Delivered


Print (restaurant,item,price) when Category is Dessert?\
print first 3 rows, first 4 columns by position?\
loc - label based selection\
iloc - position based selection


In [ ]:
df.loc[df["category"]=="Desserts",["restaurant","item","item_price","category"]].head()

,restaurant,item,item_price,category
0,Ovenfresh,Brownie,190.0,Desserts
9,Theobroma,Brownie,128.0,Desserts
14,Ovenfresh,Tiramisu (eggless),199.0,Desserts
19,Ovenfresh,Rasmalai,143.0,Desserts
37,Theobroma,Cheesecake (eggless),304.0,Desserts


In [ ]:
df.iloc[0:3,0:4]

,order_id,order_time,city,restaurant
0,101756,2024-11-27 22:23:00,Chennai,Ovenfresh
1,101396,2024-11-30 09:00:00,Pune,Haldiram's
2,100270,2024-11-08 22:32:00,Hyderabad,Veg Treat


In [ ]:
df["order_total"] = df["item_price"] * df["quantity"] + df["delivery_fee"] - df["discount"]